In [33]:
from utils import get_spark_session, cargar_tabla
from pyspark.sql.functions import desc, sum, count, round, col, lag
from pyspark.sql.window import Window

spark = get_spark_session()

### Cargar Tablas

In [12]:
hechos = cargar_tabla("fac_ventas", spark)
productos = cargar_tabla("dim_producto", spark)
clientes = cargar_tabla("dim_cliente", spark)
corresponsales = cargar_tabla("dim_corresponsal", spark)
tiempo = cargar_tabla("dim_tiempo", spark)

### Top producto más vendidos

In [13]:
top_productos = hechos.groupBy("id_producto").sum("cantidad").withColumnRenamed("sum(cantidad)", "total_vendido")\
    .join(productos, "id_producto").orderBy(desc("total_vendido")).limit(10)
top_productos.select("nombre", "total_vendido").show()

+--------------+-------------+
|        nombre|total_vendido|
+--------------+-------------+
|REPRODUCTORMP3|           18|
|  AURICULARESN|           13|
|    CABLE USBC|            6|
+--------------+-------------+



### Top 5 clientes con más pedidos

In [14]:
top_clientes = hechos.groupBy("id_cliente").count().withColumnRenamed("count", "total_pedidos")\
    .join(clientes, "id_cliente").orderBy(desc("total_pedidos")).limit(5)
top_clientes.select("nombre", "apellido", "total_pedidos").show()

+----------+--------+-------------+
|    nombre|apellido|total_pedidos|
+----------+--------+-------------+
|      JOSE|   MUNOZ|           12|
|MARIA JOSE|   PEREZ|           10|
|    ISMAEL|   GOMEZ|            9|
+----------+--------+-------------+



### Top 5 corresponsales con más pedidos

In [16]:
top_corresponsales = hechos.groupBy("id_corresponsal").count().withColumnRenamed("count", "total_pedidos")\
    .join(corresponsales, "id_corresponsal").orderBy(desc("total_pedidos")).limit(5)
top_corresponsales.select("nombre", "total_pedidos").show()

+----------+-------------+
|    nombre|total_pedidos|
+----------+-------------+
|  NANPAGOS|           13|
| RAPIPAGOS|           10|
|SERVIPAGOS|            8|
+----------+-------------+



### Total pagos diario por producto

In [17]:
df_fecha = hechos.join(tiempo, hechos.fecha == tiempo.fecha)

In [27]:
# Evita ambigüedad renombrando la columna después del join
df_fecha = hechos.join(tiempo, hechos.fecha == tiempo.fecha) \
    .drop(tiempo.fecha)  # eliminamos una de las dos columnas 'fecha'

# Ahora ya no habrá ambigüedad
pagos_diario_prod = df_fecha.groupBy("id_producto", "fecha") \
    .sum("monto_total") \
    .withColumnRenamed("sum(monto_total)", "total_pagado") \
    .join(productos, "id_producto") \
    .withColumn("total_pagado", round("total_pagado", 2))

pagos_diario_prod.select("nombre", "fecha", "total_pagado").show()


+--------------+----------+------------+
|        nombre|     fecha|total_pagado|
+--------------+----------+------------+
|REPRODUCTORMP3|2025-03-16|       90.26|
|REPRODUCTORMP3|2025-02-13|      180.52|
|REPRODUCTORMP3|2025-01-16|       90.26|
|REPRODUCTORMP3|2025-01-13|       90.26|
|REPRODUCTORMP3|2025-01-11|      361.04|
|    CABLE USBC|2025-02-13|        20.0|
|    CABLE USBC|2025-01-11|        40.0|
|  AURICULARESN|2025-04-13|      241.52|
|  AURICULARESN|2025-03-16|      120.76|
|  AURICULARESN|2025-03-13|      241.52|
|  AURICULARESN|2025-02-13|      241.52|
|  AURICULARESN|2025-01-16|      120.76|
|  AURICULARESN|2025-01-13|       603.8|
+--------------+----------+------------+



### Total pagos mensual por producto

In [20]:
pagos_mensual_prod = df_fecha.groupBy("id_producto", "mes_anio") \
    .sum("monto_total") \
    .withColumnRenamed("sum(monto_total)", "total_pagado") \
    .join(productos, "id_producto") \
    .withColumn("total_pagado", round("total_pagado", 2))
pagos_mensual_prod.select("nombre", "mes_anio", "total_pagado").show()

+--------------+--------+------------+
|        nombre|mes_anio|total_pagado|
+--------------+--------+------------+
|REPRODUCTORMP3| 2025-03|       90.26|
|REPRODUCTORMP3| 2025-01|      541.56|
|REPRODUCTORMP3| 2025-02|      180.52|
|    CABLE USBC| 2025-02|        20.0|
|    CABLE USBC| 2025-01|        40.0|
|  AURICULARESN| 2025-02|      241.52|
|  AURICULARESN| 2025-04|      241.52|
|  AURICULARESN| 2025-01|      724.56|
|  AURICULARESN| 2025-03|      362.28|
+--------------+--------+------------+



### Total pagos diario por cliente

In [28]:
pagos_diario_cliente = df_fecha.groupBy("id_cliente", "fecha") \
    .sum("monto_total") \
    .withColumnRenamed("sum(monto_total)", "total_pagado") \
    .join(clientes, "id_cliente") \
    .withColumn("total_pagado", round("total_pagado", 2))
pagos_diario_cliente.select("nombre", "apellido", "fecha", "total_pagado").show()

+----------+--------+----------+------------+
|    nombre|apellido|     fecha|total_pagado|
+----------+--------+----------+------------+
|      JOSE|   MUNOZ|2025-02-13|      200.52|
|      JOSE|   MUNOZ|2025-01-11|      401.04|
|    ISMAEL|   GOMEZ|2025-03-16|      211.02|
|    ISMAEL|   GOMEZ|2025-01-16|      211.02|
|    ISMAEL|   GOMEZ|2025-01-13|      211.02|
|MARIA JOSE|   PEREZ|2025-04-13|      241.52|
|MARIA JOSE|   PEREZ|2025-03-13|      241.52|
|MARIA JOSE|   PEREZ|2025-02-13|      241.52|
|MARIA JOSE|   PEREZ|2025-01-13|      483.04|
+----------+--------+----------+------------+



### Total pagos mensual por cliente

In [22]:
pagos_mensual_cliente = df_fecha.groupBy("id_cliente", "mes_anio") \
    .sum("monto_total") \
    .withColumnRenamed("sum(monto_total)", "total_pagado") \
    .join(clientes, "id_cliente") \
    .withColumn("total_pagado", round("total_pagado", 2))
pagos_mensual_cliente.select("nombre", "apellido", "mes_anio", "total_pagado").show()

+----------+--------+--------+------------+
|    nombre|apellido|mes_anio|total_pagado|
+----------+--------+--------+------------+
|      JOSE|   MUNOZ| 2025-01|      401.04|
|      JOSE|   MUNOZ| 2025-02|      200.52|
|    ISMAEL|   GOMEZ| 2025-03|      211.02|
|    ISMAEL|   GOMEZ| 2025-01|      422.04|
|MARIA JOSE|   PEREZ| 2025-02|      241.52|
|MARIA JOSE|   PEREZ| 2025-04|      241.52|
|MARIA JOSE|   PEREZ| 2025-01|      483.04|
|MARIA JOSE|   PEREZ| 2025-03|      241.52|
+----------+--------+--------+------------+



### Total pagos diario por corresponsal

In [29]:
pagos_diario_corr = df_fecha.groupBy("id_corresponsal", "fecha") \
    .sum("monto_total") \
    .withColumnRenamed("sum(monto_total)", "total_pagado") \
    .join(corresponsales, "id_corresponsal") \
    .withColumn("total_pagado", round("total_pagado", 2))
pagos_diario_corr.select("nombre", "fecha", "total_pagado").show()

+----------+----------+------------+
|    nombre|     fecha|total_pagado|
+----------+----------+------------+
| RAPIPAGOS|2025-03-13|      241.52|
| RAPIPAGOS|2025-01-11|      401.04|
|SERVIPAGOS|2025-03-16|      211.02|
|SERVIPAGOS|2025-01-16|      211.02|
|SERVIPAGOS|2025-01-13|      241.52|
|  NANPAGOS|2025-04-13|      241.52|
|  NANPAGOS|2025-02-13|      442.04|
|  NANPAGOS|2025-01-13|      452.54|
+----------+----------+------------+



### Total pagos mensual por corresponsal

In [24]:
pagos_mensual_corr = df_fecha.groupBy("id_corresponsal", "mes_anio") \
    .sum("monto_total") \
    .withColumnRenamed("sum(monto_total)", "total_pagado") \
    .join(corresponsales, "id_corresponsal") \
    .withColumn("total_pagado", round("total_pagado", 2))
pagos_mensual_corr.select("nombre", "mes_anio", "total_pagado").show()

+----------+--------+------------+
|    nombre|mes_anio|total_pagado|
+----------+--------+------------+
| RAPIPAGOS| 2025-03|      241.52|
| RAPIPAGOS| 2025-01|      401.04|
|SERVIPAGOS| 2025-03|      211.02|
|SERVIPAGOS| 2025-01|      452.54|
|  NANPAGOS| 2025-02|      442.04|
|  NANPAGOS| 2025-04|      241.52|
|  NANPAGOS| 2025-01|      452.54|
+----------+--------+------------+



### Variación diaria de pedidos por producto

In [35]:
df_var_dia = df_fecha.groupBy("id_producto", "fecha") \
    .sum("cantidad") \
    .withColumnRenamed("sum(cantidad)", "total_dia")

ventana_dia = Window.partitionBy("id_producto").orderBy("fecha")
df_variacion_dia = df_var_dia.withColumn("pedidos_dia_anterior", lag("total_dia").over(ventana_dia)) \
    .withColumn("variacion", col("total_dia") - col("pedidos_dia_anterior")) \
    .join(productos, "id_producto")
df_variacion_dia.select("nombre", "fecha", "total_dia", "pedidos_dia_anterior", "variacion").show()

+--------------+----------+---------+--------------------+---------+
|        nombre|     fecha|total_dia|pedidos_dia_anterior|variacion|
+--------------+----------+---------+--------------------+---------+
|REPRODUCTORMP3|2025-03-16|        2|                   4|       -2|
|REPRODUCTORMP3|2025-02-13|        4|                   2|        2|
|REPRODUCTORMP3|2025-01-16|        2|                   2|        0|
|REPRODUCTORMP3|2025-01-13|        2|                   8|       -6|
|REPRODUCTORMP3|2025-01-11|        8|                NULL|     NULL|
|    CABLE USBC|2025-02-13|        2|                   4|       -2|
|    CABLE USBC|2025-01-11|        4|                NULL|     NULL|
|  AURICULARESN|2025-04-13|        2|                   1|        1|
|  AURICULARESN|2025-03-16|        1|                   2|       -1|
|  AURICULARESN|2025-03-13|        2|                   2|        0|
|  AURICULARESN|2025-02-13|        2|                   1|        1|
|  AURICULARESN|2025-01-16|       

### Variación mensual de pedidos por corresponsal

In [34]:
df_var_mes = df_fecha.groupBy("id_corresponsal", "mes_anio") \
    .sum("cantidad") \
    .withColumnRenamed("sum(cantidad)", "total_pedidos")

ventana = Window.partitionBy("id_corresponsal").orderBy("mes_anio")
df_variacion = df_var_mes.withColumn("pedidos_mes_anterior", lag("total_pedidos").over(ventana)) \
    .withColumn("variacion", col("total_pedidos") - col("pedidos_mes_anterior")) \
    .join(corresponsales, "id_corresponsal")
df_variacion.select("nombre", "mes_anio", "total_pedidos", "pedidos_mes_anterior", "variacion").show()

+----------+--------+-------------+--------------------+---------+
|    nombre|mes_anio|total_pedidos|pedidos_mes_anterior|variacion|
+----------+--------+-------------+--------------------+---------+
| RAPIPAGOS| 2025-03|            2|                  12|      -10|
| RAPIPAGOS| 2025-01|           12|                NULL|     NULL|
|SERVIPAGOS| 2025-03|            3|                   5|       -2|
|SERVIPAGOS| 2025-01|            5|                NULL|     NULL|
|  NANPAGOS| 2025-04|            2|                   8|       -6|
|  NANPAGOS| 2025-02|            8|                   5|        3|
|  NANPAGOS| 2025-01|            5|                NULL|     NULL|
+----------+--------+-------------+--------------------+---------+

